In [1]:
import os

In [2]:
%pwd

'c:\\Users\\USER\\DataScience\\AWS-Deployment\\Image-Classification\\CT-Scan-Lung-Cancer-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\USER\\DataScience\\AWS-Deployment\\Image-Classification\\CT-Scan-Lung-Cancer-Classification'

## Config Entity

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_optimizer: str
    params_learning_rate: float
    params_loss: str
    params_metrics: list

 ## configuration manager in src config

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

        

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chest-CT-Scan-data")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(
                prepare_base_model.updated_base_model_path
            ),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
            params_optimizer=params.OPTIMIZER,
            params_learning_rate=params.LEARNING_RATE,
            params_loss=params.LOSS,
            params_metrics=params.METRICS
        )

        return training_config


## Model Training Component

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [9]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        # Load the model without reusing its previous optimizer state
        # self.model = tf.keras.models.load_model(
        #     self.config.updated_base_model_path,
        #     compile=False
        # )
        self.model = tf.keras.models.load_model(
        self.config.updated_base_model_path, compile=False)

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1.0 / 255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = (
            tf.keras.preprocessing.image.ImageDataGenerator(
                **datagenerator_kwargs
            )
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = (
                tf.keras.preprocessing.image.ImageDataGenerator(
                    rotation_range=40,
                    horizontal_flip=True,
                    width_shift_range=0.2,
                    height_shift_range=0.2,
                    shear_range=0.2,
                    zoom_range=0.2,
                    **datagenerator_kwargs
                )
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    def compile_model(self):
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.config.params_learning_rate
        )

        metric_map = {
            "accuracy": tf.keras.metrics.CategoricalAccuracy(
                name="accuracy"
            ),
            "precision": tf.keras.metrics.Precision(
                name="precision"
            ),
            "recall": tf.keras.metrics.Recall(
                name="recall"
            )
        }

        metrics = []

        for metric_name in self.config.params_metrics:
            key = metric_name.lower()

            if key not in metric_map:
                raise ValueError(
                    f"Unsupported metric: {metric_name}"
                )

            metrics.append(metric_map[key])

        self.model.compile(
            optimizer=optimizer,
            loss=self.config.params_loss,
            metrics=metrics
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        # len(generator) includes the final partial batch
        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )


## Training Pipeline

In [10]:

try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)

    training.get_base_model()
    training.train_valid_generator()
    training.compile_model()
    training.train()

except Exception as e:
    raise e


[2026-10-21 00:24:54,849: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-21 00:24:54,853: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-21 00:24:54,854: INFO: common: created directory at: artifacts]
[2026-10-21 00:24:54,854: INFO: common: created directory at: artifacts\training]
Found 68 images belonging to 2 classes.
Found 275 images belonging to 2 classes.
[2026-10-21 00:24:55,154: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.7709 - loss: 0.4884 - precision: 0.7709 - recall: 0.7709 - val_accuracy: 0.8824 - val_loss: 0.1771 - val_precision: 0.8824 - val_recall: 0.8824
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.8800 - loss: 0.2861 - precision: 0.8800 - recall: 0.8800 - val_accuracy: 1.0000 - va